# MVTec bottle pipeline

This runs the complete tensor-MD workflow on the MVTec `bottle` category. Set `MVTEC_DATA_ROOT` to the directory containing the category folders before running. The loader uses `train/good` for fitting and `test/good` plus defect folders for scoring/evaluation.

In [ ]:
import os
from pathlib import Path

from tensor_md import (
    LocationAwareTensorMahalanobisDetector,
    PatchExtractionConfig,
    load_patch_datasets,
)

MVTEC_ROOT = Path(os.environ["MVTEC_DATA_ROOT"]).expanduser().resolve()
assert (MVTEC_ROOT / "bottle" / "train" / "good").is_dir(), MVTEC_ROOT
print(MVTEC_ROOT)

In [ ]:
config = PatchExtractionConfig(
    category="bottle",
    data_root=MVTEC_ROOT,
    input_representation="cnn_features",
    cnn_backbone="ResNet50",
    cnn_layer_names=("conv3_block4_out", "conv4_block6_out"),
    cnn_pca_components=(128, 128),
    cnn_feature_patch_size=(1, 1),
    max_train_images=20,
    max_test_good_images=20,
    max_test_anomaly_images_per_type=20,
)
datasets = load_patch_datasets(config)
print("train patches:", datasets.train.patches.shape)
print("test patches:", datasets.test.patches.shape)

In [ ]:
detector = LocationAwareTensorMahalanobisDetector(
    patches_per_image=datasets.train.patches_per_image,
    covariance_shrinkage=1.0,
    iterations=6,
)

# Fit on normal training images and score the held-out MVTec test images.
detector.fit(datasets.train)
test_scores = detector.score(datasets.test)
test_scores.shape

In [ ]:
# Save score distributions, spatial heatmaps, and floating-point TIFF maps.
output_dir = Path("outputs/mvtec_bottle")
artifacts = {
    "train": detector.save_score_diagnostics(
        detector.score(datasets.train), output_dir / "train",
        grid_shape=(32, 32), image_paths=datasets.train.image_paths,
        formats=("npy", "csv", "json", "distribution", "heatmaps", "tiff"),
    ),
    "test": detector.save_score_diagnostics(
        test_scores, output_dir / "test",
        grid_shape=(32, 32), image_paths=datasets.test.image_paths,
        formats=("npy", "csv", "json", "distribution", "heatmaps", "tiff"),
    ),
}
artifacts

The diagnostics are for inspection. MVTec AU-PRO and image AUROC should still be computed with the thesis repository's official evaluator, using the generated TIFF maps and the corresponding test-image paths.